# Part 15: Reinforcement Learning

**Quick Reference for RL Algorithms**

[Back to Index](Index.ipynb)

### RL Fundamentals

**Key Concepts:**
- **Agent:** Decision maker
- **Environment:** World agent interacts with
- **State (s):** Current situation
- **Action (a):** Choice made by agent
- **Reward (r):** Feedback from environment
- **Policy (π):** Strategy for selecting actions
- **Value Function (V):** Expected return from a state

**Goal:** Maximize cumulative reward

---
## 15.1 Upper Confidence Bound (UCB)

**Concept:** Multi-armed bandit algorithm balancing exploration vs exploitation

**Formula:** UCB = average_reward + sqrt(2 * log(n) / n_i)
- n = total rounds
- n_i = times arm i was pulled

**Use Case:** A/B testing, ad selection, recommendation systems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class UCB:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)  # times each arm was pulled
        self.values = np.zeros(n_arms)  # average reward for each arm
        self.total_count = 0
    
    def select_arm(self):
        # Initially pull each arm once
        if self.total_count < self.n_arms:
            return self.total_count
        
        # Calculate UCB for each arm
        ucb_values = self.values + np.sqrt(2 * np.log(self.total_count) / (self.counts + 1e-5))
        return np.argmax(ucb_values)
    
    def update(self, arm, reward):
        self.counts[arm] += 1
        self.total_count += 1
        # Incremental average
        n = self.counts[arm]
        self.values[arm] = ((n - 1) / n) * self.values[arm] + (1 / n) * reward

# Simulation
np.random.seed(42)
true_probabilities = [0.2, 0.5, 0.8, 0.4, 0.6]  # True conversion rates
n_rounds = 1000

ucb = UCB(n_arms=len(true_probabilities))
rewards = []
selections = []

for round in range(n_rounds):
    arm = ucb.select_arm()
    reward = 1 if np.random.random() < true_probabilities[arm] else 0
    ucb.update(arm, reward)
    
    rewards.append(reward)
    selections.append(arm)

# Results
print(f"True probabilities: {true_probabilities}")
print(f"Learned values: {ucb.values}")
print(f"Arm selections: {ucb.counts}")
print(f"Total reward: {sum(rewards)}")
print(f"Best arm (should be 2): {np.argmax(ucb.values)}")

# Plot cumulative reward
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(np.cumsum(rewards))
plt.xlabel('Round')
plt.ylabel('Cumulative Reward')
plt.title('UCB: Cumulative Reward')

plt.subplot(1, 2, 2)
plt.bar(range(len(true_probabilities)), ucb.counts)
plt.xlabel('Arm')
plt.ylabel('Times Selected')
plt.title('Arm Selection Frequency')
plt.tight_layout()
plt.show()

---
## 15.2 Thompson Sampling

**Concept:** Bayesian approach to multi-armed bandit

**Method:** Sample from posterior distribution of each arm, select highest

**Advantage:** Often better than UCB, more exploration

In [ ]:
class ThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.successes = np.zeros(n_arms)  # alpha parameter
        self.failures = np.zeros(n_arms)   # beta parameter
    
    def select_arm(self):
        # Sample from Beta distribution for each arm
        samples = [np.random.beta(self.successes[i] + 1, self.failures[i] + 1) 
                   for i in range(self.n_arms)]
        return np.argmax(samples)
    
    def update(self, arm, reward):
        if reward == 1:
            self.successes[arm] += 1
        else:
            self.failures[arm] += 1

# Simulation
ts = ThompsonSampling(n_arms=len(true_probabilities))
rewards_ts = []
selections_ts = []

for round in range(n_rounds):
    arm = ts.select_arm()
    reward = 1 if np.random.random() < true_probabilities[arm] else 0
    ts.update(arm, reward)
    
    rewards_ts.append(reward)
    selections_ts.append(arm)

# Results
print(f"\nThompson Sampling Results:")
print(f"Total reward: {sum(rewards_ts)}")
estimated_probs = ts.successes / (ts.successes + ts.failures)
print(f"Estimated probabilities: {estimated_probs}")

# Compare UCB vs Thompson Sampling
plt.figure(figsize=(10, 5))
plt.plot(np.cumsum(rewards), label='UCB', alpha=0.7)
plt.plot(np.cumsum(rewards_ts), label='Thompson Sampling', alpha=0.7)
plt.xlabel('Round')
plt.ylabel('Cumulative Reward')
plt.title('UCB vs Thompson Sampling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 15.3 Q-Learning

**Concept:** Learn Q-values Q(s,a) = expected return from taking action a in state s

**Update Rule:** Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') - Q(s,a)]
- α: learning rate
- γ: discount factor

**Use Case:** Grid world, game playing

In [ ]:
import gym

class QLearning:
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.99, epsilon=0.1):
        self.q_table = np.zeros((n_states, n_actions))
        self.alpha = alpha      # learning rate
        self.gamma = gamma      # discount factor
        self.epsilon = epsilon  # exploration rate
    
    def select_action(self, state):
        # Epsilon-greedy policy
        if np.random.random() < self.epsilon:
            return np.random.randint(self.q_table.shape[1])  # explore
        return np.argmax(self.q_table[state])  # exploit
    
    def update(self, state, action, reward, next_state, done):
        # Q-learning update
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        
        self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])

# Example: FrozenLake environment
env = gym.make('FrozenLake-v1', is_slippery=False)
n_states = env.observation_space.n
n_actions = env.action_space.n

agent = QLearning(n_states, n_actions, alpha=0.8, gamma=0.95, epsilon=0.1)

# Training
n_episodes = 2000
rewards_per_episode = []

for episode in range(n_episodes):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    
    done = False
    total_reward = 0
    
    while not done:
        action = agent.select_action(state)
        result = env.step(action)
        next_state, reward, done = result[0], result[1], result[2]
        
        agent.update(state, action, reward, next_state, done)
        
        state = next_state
        total_reward += reward
    
    rewards_per_episode.append(total_reward)

# Evaluate
print(f"\nQ-Learning Results:")
print(f"Average reward (last 100 episodes): {np.mean(rewards_per_episode[-100:]):.3f}")

# Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(np.convolve(rewards_per_episode, np.ones(100)/100, mode='valid'))
plt.xlabel('Episode')
plt.ylabel('Average Reward (100 episodes)')
plt.title('Q-Learning: Learning Curve')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nLearned Q-table (first 5 states):\n{agent.q_table[:5]}")

---
## 15.4 Deep Q-Network (DQN)

**Concept:** Use neural network to approximate Q-values

**Key Innovations:**
- Experience replay: Store and sample past experiences
- Target network: Separate network for stable targets

**Use Case:** High-dimensional state spaces (images, etc.)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)

class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon=1.0, epsilon_decay=0.995):
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = 0.01
        
        # Networks
        self.policy_net = DQN(state_dim, action_dim)
        self.target_net = DQN(state_dim, action_dim)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
    
    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.action_dim)
        
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()
    
    def train(self, batch_size=64):
        if len(self.buffer) < batch_size:
            return
        
        # Sample batch
        batch = self.buffer.sample(batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones)
        
        # Compute Q-values
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # Compute target Q-values
        with torch.no_grad():
            next_q_values = self.target_net(next_states).max(1)[0]
            targets = rewards + (1 - dones) * self.gamma * next_q_values
        
        # Compute loss and update
        loss = nn.MSELoss()(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def update_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

# Training loop (simplified)
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

agent = DQNAgent(state_dim, action_dim)
n_episodes = 500
target_update_freq = 10

episode_rewards = []

for episode in range(n_episodes):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    
    done = False
    total_reward = 0
    
    while not done:
        action = agent.select_action(state)
        result = env.step(action)
        next_state, reward, done = result[0], result[1], result[2]
        
        agent.buffer.push(state, action, reward, next_state, done)
        agent.train()
        
        state = next_state
        total_reward += reward
    
    episode_rewards.append(total_reward)
    
    # Update target network
    if episode % target_update_freq == 0:
        agent.update_target_network()
    
    if episode % 50 == 0:
        print(f"Episode {episode}, Avg Reward: {np.mean(episode_rewards[-50:]):.2f}, Epsilon: {agent.epsilon:.3f}")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.3)
plt.plot(np.convolve(episode_rewards, np.ones(50)/50, mode='valid'), linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('DQN: Learning Curve')
plt.grid(True, alpha=0.3)
plt.show()

---
### Quick Reference

**Algorithm Comparison:**

| Algorithm | Type | State Space | Action Space | Use Case |
|-----------|------|-------------|--------------|----------|
| UCB | Bandit | Single | Finite | A/B testing |
| Thompson Sampling | Bandit | Single | Finite | A/B testing |
| Q-Learning | Value-based | Discrete | Discrete | Simple games |
| DQN | Value-based | Continuous | Discrete | Atari games |
| Policy Gradient | Policy-based | Continuous | Continuous | Robotics |
| Actor-Critic | Hybrid | Continuous | Continuous | Complex tasks |

**Key Hyperparameters:**
- **Learning rate (α):** 0.001-0.1 (smaller for DQN)
- **Discount factor (γ):** 0.9-0.99 (higher for long-term planning)
- **Exploration rate (ε):** Start 1.0, decay to 0.01

**When to use:**
- Simple A/B testing → UCB or Thompson Sampling
- Discrete state/action → Q-Learning
- High-dimensional states → DQN
- Continuous actions → Policy Gradient or Actor-Critic
- Sample efficiency matters → Model-based RL